In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

In [ ]:
# Problem data.
n = 100
m = 100
p = 3
ratings_per_user = 5 # number of movies that each user rated

In [ ]:
# Generate random ground truth ratings.
np.random.seed(0)
U_nom = np.random.uniform(-1, 1, (n, p)) # user matrix
M_nom = np.random.uniform(0, 1, (m, p)) # movie matrix
R_nom = U_nom @ M_nom.T / p # rating matrix

# Plot rating matrix.
plt.figure()
plt.imshow(R_nom, vmin=-1, vmax=1)
plt.colorbar(label='rating')
plt.xlabel('movie')
plt.ylabel('user')
plt.title('Ground truth ratings')
plt.show()

In [ ]:
# Rated movies for each user.
np.random.seed(0)
rating_mask = np.zeros((n, m))
for i in range(n):
    for j in np.random.choice(range(m), ratings_per_user, replace=False):
        rating_mask[i, j] = 1

# Plot masked rating matrix.
plt.figure()
R_masked = np.ma.masked_where(rating_mask == 0, R_nom)
plt.imshow(R_masked, vmin=-1, vmax=1)
plt.colorbar(label='rating')
plt.xlabel('movie')
plt.ylabel('user')
plt.title('Observed ratings')
plt.show()

In [ ]:
# Initialize biconvex method.
tol = .001 # convergence tolerance
M_est = np.ones((m, p)) / 2 # initial estimate for the movie types
prev_error = np.inf

# Run biconvex method.
while True:

    # Improve user matrix estimate.
    U = cp.Variable((n, p))
    constraints = [U >= -1, U <= 1]
    residuals = R_nom - U @ M_est.T / p
    objective = cp.sum_squares(cp.multiply(rating_mask, residuals))
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve()
    U_est = U.value

    # Improve movie matrix estimate.
    M = cp.Variable((m, p))
    constraints = [M >= 0, M <= 1]
    residuals = R_nom - U_est @ M.T / p
    objective = cp.sum_squares(cp.multiply(rating_mask, residuals))
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve()
    M_est = M.value

    # Convergence check.
    print("Estimation error", prob.value)
    if prev_error - prob.value < tol: # converged
        R_est = U_est @ M_est.T / p
        break
    else: # not converged
        prev_error = prob.value

In [ ]:
# Plot rating matrix estimate.
plt.figure()
plt.imshow(R_est, vmin=-1, vmax=1)
plt.colorbar(label='rating')
plt.xlabel('movie')
plt.ylabel('user')
plt.title('Reconstructed ratings')
plt.show()

In [ ]:
# Plot estimation error.
plt.figure()
R_err = np.abs(R_nom - R_est)
plt.imshow(R_err, vmin=0, vmax=2)
plt.colorbar(label='rating')
plt.xlabel('movie')
plt.ylabel('user')
plt.title('Reconstruction error')
plt.show()